# SIC8 | 02 — Historical Canonical Mapping

**Goal:** Convert the IBM historical transactions into a typed, source-aware **mapping candidate**. Keep the original CSV unchanged and isolate fraud ground-truth labels from detection inputs.

**Output:** `transactions_candidate` + `ground_truth_candidate` + quality checks. **No Delta or S3 writes in this notebook.**

## 01 — Settings
Read one CSV file, not the entire baseline folder. Start with `SAMPLE`; use `FULL` only after reviewing the sample results.

In [0]:
from pyspark.sql import functions as F
import time

RAW_PATH = "/Volumes/workspace/sic8_sources/historical_raw"
TRANSACTIONS_FILE = f"{RAW_PATH}/credit_card_transactions-ibm_v2.csv"
MODE = "SAMPLE"            # SAMPLE or FULL
SAMPLE_ROWS = 5000
SOURCE_VERSION = "IBM_V2"  # Project-local source label, not an AWS object version

RAW_COLUMNS = [
    "User", "Card", "Year", "Month", "Day", "Time", "Amount",
    "Use Chip", "Merchant Name", "Merchant City", "Merchant State",
    "Zip", "MCC", "Errors?", "Is Fraud?"
]

print("Mapping mode:", MODE)
print("Source:", TRANSACTIONS_FILE)

Mapping mode: SAMPLE
Source: /Volumes/workspace/sic8_sources/historical_raw/credit_card_transactions-ibm_v2.csv


## 02 — Load the source
Use raw strings so invalid values remain visible instead of being silently coerced to null.

In [0]:
transactions_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("enforceSchema", "false")
    .csv(TRANSACTIONS_FILE)
)

if transactions_raw.columns != RAW_COLUMNS:
    raise ValueError(f"Unexpected CSV columns: {transactions_raw.columns}")
if MODE not in ("SAMPLE", "FULL"):
    raise ValueError("MODE must be SAMPLE or FULL")

source_df = (transactions_raw.limit(SAMPLE_ROWS)
             if MODE == "SAMPLE" else transactions_raw)
print("Source columns:", len(source_df.columns))
print("Read-only mode; no output tables are created.")

Source columns: 15
Read-only mode; no output tables are created.


## 03 — Identity and lineage
Namespace User/Card IDs to avoid mixing historical identities with synthetic users. A row fingerprint is useful for auditing, **not** a unique transaction ID: two distinct source rows can have identical content.

In [0]:
def has_value(name):
    return F.col(name).isNotNull() & (F.trim(F.col(name)) != "")

# Use the raw transaction fields, excluding source fraud truth, for the fingerprint.
raw_fields = [F.col(name).alias(f"f{i:02d}")
              for i, name in enumerate(RAW_COLUMNS[:-1], start=1)]

identity_df = (
    source_df
    .withColumn("source_record_fingerprint", F.sha2(F.to_json(F.struct(*raw_fields)), 256))
    .withColumn("_user", F.trim(F.col("User")))
    .withColumn("_card", F.trim(F.col("Card")))
    .withColumn("customer_id", F.when(
        F.col("_user").rlike("^[0-9]+$"), F.concat(F.lit("IBM_USER_"), F.col("_user"))))
    .withColumn("card_id", F.when(
        F.col("_user").rlike("^[0-9]+$") & F.col("_card").rlike("^[0-9]+$"),
        F.concat(F.lit("IBM_CARD_"), F.col("_user"), F.lit("_"), F.col("_card"))))
    .withColumn("source_merchant_id", F.when(has_value("Merchant Name"),
        F.trim(F.col("Merchant Name"))))
)

print("Historical event_id and transaction_id are still unresolved.")

Historical event_id and transaction_id are still unresolved.


## 04 — Amounts and local time
Use `DECIMAL(18,2)` for money. The CSV has no authoritative timezone, so preserve the parsed **local timestamp (TIMESTAMP_NTZ)** rather than claiming it is UTC.

In [0]:
typed_df = (
    identity_df
    .withColumn(
        "amount",
        F.expr("try_cast(regexp_replace(trim(`Amount`), '[$,]', '') AS DECIMAL(18,2))")
    )
    .withColumn("currency", F.lit("USD"))
    .withColumn("_local_time_text", F.expr("""
        concat(lpad(trim(`Year`),4,'0'), '-',
               lpad(trim(`Month`),2,'0'), '-',
               lpad(trim(`Day`),2,'0'), ' ', trim(`Time`))
    """))
    .withColumn("source_event_time_local", F.expr(
        "try_cast(_local_time_text AS TIMESTAMP_NTZ)"))
    .withColumn("event_time_timezone_status", F.lit("UNKNOWN"))
    .withColumn("amount_sign", F.when(F.col("amount") < 0, "NEGATIVE")
        .when(F.col("amount") == 0, "ZERO")
        .when(F.col("amount") > 0, "POSITIVE")
        .otherwise("INVALID"))
)

print("Types set: amount=DECIMAL(18,2), local time=TIMESTAMP_NTZ")

Types set: amount=DECIMAL(18,2), local time=TIMESTAMP_NTZ


## 05 — Payment and location semantics
`Use Chip` is a payment method, **not** a confirmed POS/ATM/e-commerce channel. Merchant City/ZIP cannot prove exact GPS; unknown location remains null.

In [0]:
mapped_df = (
    typed_df
    .withColumn("payment_method", F.when(F.col("Use Chip") == "Swipe Transaction", "SWIPE")
        .when(F.col("Use Chip") == "Chip Transaction", "CHIP")
        .when(F.col("Use Chip") == "Online Transaction", "ONLINE")
        .otherwise("UNKNOWN"))
    .withColumn(
    "channel",
    F.when(F.col("payment_method") == "ONLINE", "ECOMMERCE")
     .when(F.col("payment_method").isin("SWIPE", "CHIP"), "POS")
     .otherwise("UNKNOWN")
    )
    .withColumn(
    "channel_quality",
    F.when(F.col("payment_method") == "ONLINE", "INFERRED")
     .when(F.col("payment_method").isin("SWIPE", "CHIP"), "ASSUMED")
     .otherwise("UNKNOWN")
    )
    .withColumn("card_present", F.when(F.col("payment_method").isin("SWIPE", "CHIP"), True)
        .when(F.col("payment_method") == "ONLINE", False)
        .otherwise(F.lit(None).cast("boolean")))
    .withColumn("card_presence_quality", F.lit("INFERRED_FROM_PAYMENT_METHOD"))
    .withColumn("transaction_type", F.lit("UNKNOWN"))
    .withColumn("merchant_city", F.col("Merchant City"))
    .withColumn("merchant_region_raw", F.col("Merchant State"))
    .withColumn("merchant_postal_code", F.col("Zip"))
    .withColumn("mcc", F.col("MCC"))
    .withColumn("latitude", F.lit(None).cast("double"))
    .withColumn("longitude", F.lit(None).cast("double"))
    .withColumn("location_type", F.when(F.col("payment_method") == "ONLINE",
        "ONLINE_NO_PHYSICAL_LOCATION").otherwise("UNKNOWN"))
    .withColumn("outcome_status", F.lit("UNKNOWN"))
    .withColumn("errors_raw", F.col("Errors?"))
    .withColumn("source_system", F.lit("IBM_KAGGLE_HISTORICAL"))
    .withColumn("ingestion_mode", F.lit("BATCH"))
    .withColumn("source_dataset_version", F.lit(SOURCE_VERSION))
    .withColumn("mapping_version", F.lit("candidate-0.1"))
    )

print("Channel and GPS remain unverified; no locations were fabricated.")

Channel and GPS remain unverified; no locations were fabricated.


## 06 — Row checks
Flag missing/invalid required data; do not delete rows or fill unknown values. A missing merchant state, postal code, or error note is not automatically a rejected transaction.

In [0]:
checked_df = (
    mapped_df
    .withColumn("bad_identity", F.col("customer_id").isNull() | F.col("card_id").isNull())
    .withColumn("bad_amount", F.col("amount").isNull())
    .withColumn("bad_local_time", F.col("source_event_time_local").isNull())
    .withColumn("unknown_payment_method", F.col("payment_method") == "UNKNOWN")
    .withColumn("quality_status", F.when(
        F.col("bad_identity") | F.col("bad_amount") | F.col("bad_local_time"),
        "REVIEW").otherwise("PASS"))
    .withColumn("quality_reason", F.concat_ws(" | ",
        F.when(F.col("bad_identity"), "BAD_IDENTITY"),
        F.when(F.col("bad_amount"), "INVALID_AMOUNT"),
        F.when(F.col("bad_local_time"), "INVALID_LOCAL_TIME")))
)

print("Quality status is a preliminary mapping check, not a production approval.")

Quality status is a preliminary mapping check, not a production approval.


## 07 — Keep fraud labels separate
`Is Fraud?` is **source ground truth**, not a fraud-detection feature. These candidate datasets share a fingerprint for inspection only; evaluation requires an approved unique event key.

In [0]:
TRANSACTION_FIELDS = [
    "source_system", "ingestion_mode", "source_dataset_version", "mapping_version",
    "source_record_fingerprint", "customer_id", "card_id", "source_merchant_id",
    "source_event_time_local", "event_time_timezone_status", "amount", "amount_sign",
    "currency", "channel", "payment_method", "card_present", "card_presence_quality",
    "transaction_type", "mcc", "merchant_city", "merchant_region_raw",
    "merchant_postal_code", "latitude", "longitude", "location_type",
    "outcome_status", "errors_raw", "quality_status", "quality_reason"
]

transactions_candidate = checked_df.select(*TRANSACTION_FIELDS)

ground_truth_candidate = checked_df.select(
    "source_record_fingerprint",
    F.when(F.col("Is Fraud?") == "Yes", True)
     .when(F.col("Is Fraud?") == "No", False)
     .otherwise(F.lit(None).cast("boolean")).alias("ground_truth_label"),
    F.lit("IBM_SOURCE_LABEL").alias("label_source")
)

assert "Is Fraud?" not in transactions_candidate.columns
assert "ground_truth_label" not in transactions_candidate.columns
print("Fraud label isolated from transaction features.")
print("Candidate transactions:", len(transactions_candidate.columns), "fields")

Fraud label isolated from transaction features.
Candidate transactions: 29 fields


## 08 — Mapping quality summary
Calculate a small set of acceptance metrics in one Spark action. `FULL` reads the full CSV; run it only after checking the sample.

In [0]:
start = time.perf_counter()
metrics = checked_df.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("bad_identity").cast("long")).alias("bad_identity"),
    F.sum(F.col("bad_amount").cast("long")).alias("bad_amount"),
    F.sum(F.col("bad_local_time").cast("long")).alias("bad_local_time"),
    F.sum(F.col("unknown_payment_method").cast("long")).alias("unknown_method"),
    F.sum((F.col("amount") < 0).cast("long")).alias("negative_amount"),
    F.sum((F.col("quality_status") == "REVIEW").cast("long")).alias("review_rows"),
    F.sum((F.col("Is Fraud?") == "Yes").cast("long")).alias("fraud_yes"),
    F.sum((F.col("Is Fraud?") == "No").cast("long")).alias("fraud_no"),
).first().asDict()

print("Mode:", MODE, "| seconds:", round(time.perf_counter() - start, 2))
display(spark.createDataFrame(
    [(k, str(v)) for k, v in metrics.items()], ["metric", "value"]))

if MODE == "SAMPLE":
    print("Note: the first rows are not a representative random sample.")

Mode: SAMPLE | seconds: 4.76


metric,value
rows,5000
bad_identity,0
bad_amount,0
bad_local_time,0
unknown_method,0
negative_amount,174
review_rows,0
fraud_yes,13
fraud_no,4987


Note: the first rows are not a representative random sample.


## 09 — Inspect mapped records
Preview only business-safe fields. No source fraud labels, raw card numbers, or invented GPS values are shown.

In [0]:
display(transactions_candidate.select(
    "customer_id", "card_id", "source_event_time_local", "amount", "currency",
    "payment_method", "channel", "location_type", "quality_status", "quality_reason"
).limit(10))

print("Ground-truth fields:", ground_truth_candidate.columns)
print("No tables or external storage objects were changed.")

customer_id,card_id,source_event_time_local,amount,currency,payment_method,channel,location_type,quality_status,quality_reason
IBM_USER_0,IBM_CARD_0_0,2002-09-01T06:21:00.000,134.09,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-01T06:42:00.000,38.48,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-02T06:22:00.000,120.34,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-02T17:45:00.000,128.95,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-03T06:23:00.000,104.71,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-03T13:53:00.000,86.19,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-04T05:51:00.000,93.84,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-04T06:09:00.000,123.50,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-05T06:14:00.000,61.72,USD,SWIPE,POS,UNKNOWN,PASS,
IBM_USER_0,IBM_CARD_0_0,2002-09-05T09:35:00.000,57.10,USD,SWIPE,POS,UNKNOWN,PASS,


Ground-truth fields: ['source_record_fingerprint', 'ground_truth_label', 'label_source']
No tables or external storage objects were changed.


## 10 — Check provisional identity collisions (SAMPLE only)
A fingerprint collision often means identical source rows, **not necessarily a retry**. Never deduplicate transactions solely on this value.

In [0]:
if MODE == "SAMPLE":
    repeated_fingerprints = (
        checked_df.groupBy("source_record_fingerprint")
        .count()
        .filter(F.col("count") > 1)
    )
    print("Repeated content fingerprints (sample):", repeated_fingerprints.count())
    display(repeated_fingerprints.orderBy(F.desc("count")).limit(5))
else:
    print("Skipped expensive full-dataset fingerprint grouping.")

Repeated content fingerprints (sample): 0


source_record_fingerprint,count


## 11 — Decision log / next step

**Confirmed mapping choices:** Preserve signed USD amount, source timestamps without timezone claims, composite card identity, source location uncertainty, and isolated fraud labels.

**Before writing production Silver:** Agree on a reproducible **per-record ID** for historical CSV rows, verify full-source profile and malformed-row handling, and define event-time normalization/replay policy. `source_record_fingerprint` is **not** sufficient as a unique event ID. Later notebooks will handle reference joins, streaming events, and Delta writes.